# Edge check: voice-assessment-agent (Python)
Expected edges come from the **CPython `ast` (`lang-ref/python/edges.py`)**, not tree-sitter. They are compared with code-review-graph (CRG) `graph.db`, opened read-only, and with the tree-sitter module (`core-module/edges.py`).

| Edge | Reference | Match key |
|---|---|---|
| CONTAINS | named functions and classes (the same references as the function and class checks) | file + start line + name (Python: decorator line, else `def` line) |
| INHERITS | `class X(Base)` (keywords such as `metaclass=` skipped) | file + class start line + parent name |
| IMPORTS_FROM | one row per `import a` name and per `from m import …` statement | file + line |
| CALLS | `f()`, `a.b()` | file + line + name (line of the name, else line where the call starts) |

- **Recall** = expected edges found. **Precision** = reported edges that the reference also has.
- CALLS = call sites by name. It doesn't check which function is reached.
- REFERENCES edges (CRG has them here) are not checked yet.
- Each run is saved to `results/voice-assessment-agent/checks/<run>/` and logged in `results/voice-assessment-agent/checks.csv`.

In [1]:
import json, sqlite3, subprocess, pathlib, datetime, sys
import pandas as pd
pd.set_option('display.max_colwidth', 90)
ROOT = pathlib.Path('..').resolve()
APP, LANG = 'voice-assessment-agent', 'python'
REPO = ROOT / 'apps' / APP
CRG_RUN = 'iter-005_2026-11-17_2251'
DB = ROOT / f'results/{APP}/runs/{CRG_RUN}/crg/graph.db'
RUN = 'edges_' + datetime.datetime.now().strftime('%Y-%m-%d_%H%M')
OUT = ROOT / f'results/{APP}/checks/{RUN}'; OUT.mkdir(parents=True, exist_ok=True)
REF_COMMIT = subprocess.run(['git','rev-parse','--short','HEAD'], cwd=REPO, capture_output=True, text=True).stdout.strip()
EXT = {'javascript': ['*.js','*.jsx','*.mjs','*.cjs'], 'python': ['*.py']}[LANG]
files = subprocess.run(['git','ls-files', *EXT], cwd=REPO, capture_output=True, text=True, check=True).stdout.split()
print(RUN, '| repo commit', REF_COMMIT, '| CRG run', CRG_RUN, '| files', len(files))

edges_2026-11-20_1245 | repo commit 4e24898 | CRG run iter-005_2026-11-17_2251 | files 72


## 1. Reference

In [2]:
out = subprocess.run([sys.executable, str(ROOT/'lang-ref/python/edges.py'), str(REPO)], input='\n'.join(files), capture_output=True, text=True, check=True).stdout
rows = [json.loads(l) for l in out.splitlines()]
errors = sum(r['e'] == 'error' for r in rows)

In [3]:
(OUT/'ref_edges.jsonl').write_text('\n'.join(json.dumps(r) for r in rows))
KEY = {'CONTAINS':'contains','INHERITS':'inherits','IMPORTS_FROM':'import','CALLS':'call'}
R = {e: pd.DataFrame([r for r in rows if r['e'] == k], columns=None) for e, k in KEY.items()}
for e in R:
    if R[e].empty: R[e] = pd.DataFrame(columns=['file','line','child','sup','name','start','def_line'])
print('parse errors:', errors)
pd.Series({e: len(d) for e, d in R.items()}, name='reference rows')

parse errors: 0


CONTAINS         554
INHERITS          56
IMPORTS_FROM     498
CALLS           3452
Name: reference rows, dtype: int64

## 2. code-review-graph edges

In [4]:
con = sqlite3.connect(f'file:{DB}?mode=ro', uri=True)
P = str(REPO) + '/'
rel = lambda s: s.str.replace(P, '', regex=False)
nodes = pd.read_sql('select qualified_name, name, line_start, kind node_kind from nodes', con); nodes['qualified_name'] = rel(nodes.qualified_name)
def crg(kind):
    d = pd.read_sql('select source_qualified src, target_qualified tgt, file_path file, line from edges where kind=?', con, params=(kind,))
    for c in ['src','tgt','file']: d[c] = rel(d[c])
    return d[d.file.isin(set(files))].reset_index(drop=True)
last = lambda t: t.split('::')[-1].split('.')[-1].split(':')[0]
c = crg('CONTAINS').merge(nodes, left_on='tgt', right_on='qualified_name', how='left')
c = c[c.node_kind.isin(['Function','Test','Class'])].rename(columns={'line_start':'child_line','name':'child'}).reset_index(drop=True)
i = crg('INHERITS').merge(nodes[['qualified_name','line_start']], left_on='src', right_on='qualified_name', how='left'); i['sup'] = i.tgt.map(last)
k = crg('CALLS'); k['name'] = k.tgt.map(last)
G = {'CONTAINS': c, 'INHERITS': i, 'IMPORTS_FROM': crg('IMPORTS_FROM'), 'CALLS': k}
print('CRG REFERENCES edges in these files (not checked):', len(crg('REFERENCES')))
pd.Series({e: len(d) for e, d in G.items()}, name='CRG rows (code files; CONTAINS -> functions/classes)')

CRG REFERENCES edges in these files (not checked): 418


CONTAINS         554
INHERITS          56
IMPORTS_FROM     515
CALLS           3401
Name: CRG rows (code files; CONTAINS -> functions/classes), dtype: int64

Matching is multiset: each edge is used at most once. A second key is tried only for rows still unmatched.

In [5]:
def match(ref, got, passes):
    ref = ref.copy(); got = got.copy(); ref['hit'] = False; got['hit'] = False
    for rk, gk in passes:
        if ref.empty or got.empty: break
        rr = ref[~ref.hit].copy(); gg = got[~got.hit].copy()
        rr['_n'] = rr.groupby(rk).cumcount(); gg['_n'] = gg.groupby(gk).cumcount()
        j = rr.reset_index().merge(gg.reset_index(), left_on=rk+['_n'], right_on=gk+['_n'], suffixes=('','_g'))
        ref.loc[j['index'], 'hit'] = True; got.loc[j['index_g'], 'hit'] = True
    return ref, got
SUMMARY, RES = [], {}
def report(label, ref, got):
    m, e = int(ref.hit.sum()), int(got.hit.sum())
    SUMMARY.append({'edge': label, 'expected': len(ref), 'reported': len(got), 'matched': m, 'missed': len(ref)-m,
                    'extra': len(got)-e, 'recall %': round(100*m/len(ref),1) if len(ref) else None,
                    'precision %': round(100*e/len(got),1) if len(got) else None})
    RES[label] = (ref, got)
    ref[~ref.hit].drop(columns='hit').to_csv(OUT/f'{label.replace(" ","_")}_missed.csv', index=False)
    got[~got.hit].drop(columns='hit').to_csv(OUT/f'{label.replace(" ","_")}_extra.csv', index=False)
CONT2 = [(['file','def_line','child'], ['file','child_line','child'])]
CRG_KEYS = {'CONTAINS': [(['file','line','child'], ['file','child_line','child'])] + CONT2,
            'INHERITS': [(['file','line','sup'], ['file','line_start','sup'])] + [(['file','def_line','sup'], ['file','line_start','sup'])],
            'IMPORTS_FROM': [(['file','line'], ['file','line'])],
            'CALLS': [(['file','line','name'], ['file','line','name']), (['file','start','name'], ['file','line','name'])]}

## 3. CRG vs reference

In [6]:
for e in G: report(e, *match(R[e], G[e], CRG_KEYS[e]))
pd.DataFrame(SUMMARY).set_index('edge')

,expected,reported,matched,missed,extra,recall %,precision %
edge,,,,,,,
CONTAINS,554,554,554,0,0,100.0,100.0
INHERITS,56,56,56,0,0,100.0,100.0
IMPORTS_FROM,498,515,458,40,57,92.0,88.9
CALLS,3452,3401,3399,53,2,98.5,99.9


### Why CRG misses or adds edges

In [7]:
ref, got = RES['IMPORTS_FROM']; m = ref[~ref.hit]
display(m.name.eq('__future__').map({True:'from __future__ import ...'}).fillna('other').value_counts().rename('IMPORTS missed'))
x = got[~got.hit]
display(x.tgt.str.endswith('__init__.py').map({True:'package __init__.py added beside the module'}).fillna('other').value_counts().rename('IMPORTS extra'))
display(RES['CONTAINS'][0][~RES['CONTAINS'][0].hit].head(10))
display(RES['CALLS'][1][~RES['CALLS'][1].hit].head(10))

name
from __future__ import ...    40
Name: IMPORTS missed, dtype: int64

tgt
other    57
Name: IMPORTS extra, dtype: int64

,file,e,line,def_line,parent,child,kind,hit


,src,tgt,file,line,name,hit
1884,scripts/selftest.py::_drive,scripts/selftest.py::CaptureObserver.rate_limited,scripts/selftest.py,319,rate_limited,False
2010,scripts/selftest.py::main,scripts/selftest.py::CaptureObserver.rate_limited,scripts/selftest.py,529,rate_limited,False


In [8]:
for label in G:
    print(label, 'extra (CRG only), first rows:')
    display(RES[label][1][~RES[label][1].hit].head(5))

CONTAINS extra (CRG only), first rows:


,src,tgt,file,line,qualified_name,child,child_line,node_kind,hit


INHERITS extra (CRG only), first rows:


,src,tgt,file,line,qualified_name,line_start,sup,hit


IMPORTS_FROM extra (CRG only), first rows:


,src,tgt,file,line,hit
8,app/agent/pipeline.py,app/agent/prompts.py,app/agent/pipeline.py,72,False
9,app/agent/pipeline.py,app/agent/tools.py,app/agent/pipeline.py,73,False
58,app/api/assessment.py,app/services/assessment_service.py,app/api/assessment.py,12,False
75,app/api/calls.py,app/services/call_service.py,app/api/calls.py,17,False
76,app/api/calls.py,app/services/candidate_resolver.py,app/api/calls.py,18,False


CALLS extra (CRG only), first rows:


,src,tgt,file,line,name,hit
1884,scripts/selftest.py::_drive,scripts/selftest.py::CaptureObserver.rate_limited,scripts/selftest.py,319,rate_limited,False
2010,scripts/selftest.py::main,scripts/selftest.py::CaptureObserver.rate_limited,scripts/selftest.py,529,rate_limited,False


## 4. Tree-sitter module (`core-module/edges.py`)
Contains comes from the module's own function finder (`extract.py`) and class finder (`classes.py`).

In [9]:
sys.path.insert(0, str(ROOT/'core-module'))
import importlib, edges as ME; importlib.reload(ME)
M = {k: [] for k in KEY.values()}
for f in files:
    for k2, v in ME.module_edges(REPO, f, LANG).items(): M[k2] += v
with open(OUT/'module_edges.jsonl','w') as fh:
    for k2, v in M.items():
        for r in v: fh.write(json.dumps({'e': k2, **r}) + '\n')
M = {e: pd.DataFrame(M[k2]) if M[k2] else pd.DataFrame(columns=R[e].columns) for e, k2 in KEY.items()}
MOD_KEYS = {'CONTAINS': [(['file','line','child'], ['file','line','child'])] + [(['file','def_line','child'], ['file','line','child'])],
            'INHERITS': [(['file','line','sup'], ['file','line','sup'])],
            'IMPORTS_FROM': [(['file','line'], ['file','line'])],
            'CALLS': [(['file','line','name'], ['file','line','name']), (['file','start','name'], ['file','start','name'])]}
for e in M: report(f'{e} (module)', *match(R[e], M[e], MOD_KEYS[e]))
S = pd.DataFrame(SUMMARY).set_index('edge')
pd.DataFrame({e: {'expected': S.loc[e,'expected'], 'CRG recall %': S.loc[e,'recall %'], 'module recall %': S.loc[f'{e} (module)','recall %'],
                  'CRG precision %': S.loc[e,'precision %'], 'module precision %': S.loc[f'{e} (module)','precision %'],
                  'CRG missed': S.loc[e,'missed'], 'module missed': S.loc[f'{e} (module)','missed'],
                  'CRG extra': S.loc[e,'extra'], 'module extra': S.loc[f'{e} (module)','extra']} for e in G}).T

,expected,CRG recall %,module recall %,CRG precision %,module precision %,CRG missed,module missed,CRG extra,module extra
CONTAINS,554.0,100.0,100.0,100.0,100.0,0.0,0.0,0.0,0.0
INHERITS,56.0,100.0,100.0,100.0,100.0,0.0,0.0,0.0,0.0
IMPORTS_FROM,498.0,92.0,100.0,88.9,100.0,40.0,0.0,57.0,0.0
CALLS,3452.0,98.5,100.0,99.9,100.0,53.0,0.0,2.0,0.0


In [10]:
for e in G:
    ref, got = RES[f'{e} (module)']
    if (~ref.hit).any() or (~got.hit).any():
        print(e, '- module missed / extra:'); display(ref[~ref.hit].head(5)); display(got[~got.hit].head(5))

## 5. Save

In [11]:
S = pd.DataFrame(SUMMARY).set_index('edge'); S.to_csv(OUT/'summary.csv')
log = ROOT / f'results/{APP}/checks.csv'
row = S.reset_index().rename(columns={'expected':'expected (javac)','reported':'CRG','extra':'CRG extra'})
row = row.assign(run=RUN, repo_commit=REF_COMMIT, crg_run=CRG_RUN, reference=('TypeScript compiler (js-ref/edges.mjs)' if LANG=='javascript' else 'CPython ast (lang-ref/python/edges.py)'))
row.to_csv(log, mode='a', header=not log.exists(), index=False)
print('saved to', OUT.relative_to(ROOT), '| logged in', log.relative_to(ROOT))
S

saved to results/voice-assessment-agent/checks/edges_2026-11-20_1245 | logged in results/voice-assessment-agent/checks.csv


,expected,reported,matched,missed,extra,recall %,precision %
edge,,,,,,,
CONTAINS,554,554,554,0,0,100.0,100.0
INHERITS,56,56,56,0,0,100.0,100.0
IMPORTS_FROM,498,515,458,40,57,92.0,88.9
CALLS,3452,3401,3399,53,2,98.5,99.9
CONTAINS (module),554,554,554,0,0,100.0,100.0
INHERITS (module),56,56,56,0,0,100.0,100.0
IMPORTS_FROM (module),498,498,498,0,0,100.0,100.0
CALLS (module),3452,3452,3452,0,0,100.0,100.0
